In [ ]:
pip install --upgrade langchain langchain-core langchain-community langchain-ollama

In [ ]:
pip install langchain-community --upgrade

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

# Load the Document (PDF)

In [23]:
loader = PyMuPDFLoader("/Users/yashshakya787/genAI/data/customer_support_kb.pdf")
docs = loader.load()
print(len(docs))
print(f"\nPage 1 preview:\n{docs[0].page_content[:300]}")

2

Page 1 preview:
TOPIC: Order & Delivery
Q: My order has not arrived yet.
A: Orders typically take 3-5 business days.
Track your order using the tracking ID sent to your email.
If it has been more than 7 days, contact us with your Order ID.
Q: How do I track my order?
A: Visit our website and go to My Orders section


# 2nd step :- split the documents

In [24]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50,
    separators    = ["\n\n", "\n", "Q:", "A:", " "]
)
chunks = splitter.split_documents(docs)
print(f" Total chunks: {len(chunks)}")

 Total chunks: 8


# OllamaEmbeddings + Create the vector database (ChromaDB)

In [25]:
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore_CDB = Chroma.from_documents(
    documents = chunks,
    embedding= embedding,
    collection_name="customer_support_systemx",
    persist_directory=" ../data/chroma__db"
)

print(f"   Vectors stored: {vectorstore_CDB._collection.count()}")

   Vectors stored: 160


#  OllamaLLM + RAG Chain

In [26]:
#create the retriver from the chromadb

retriever = vectorstore_CDB.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 3}
)

# create the LLm
llm = Ollama(
    model = 'llama3.1:8b',
    temperature= 0.2
)

# create the chatpromptemplate
prompt = ChatPromptTemplate.from_template(""" 
You are a helpful customer support agent.
Use ONLY the context below to answer the question.
If answer not found, say: "I don't have info on that. Let me connect you with a human agent."
Always end with: "Is there anything else I can help you with?"

Context:
{context}

Customer Question: {question}

Support Agent Response:
""")


# helper
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL RAG chain with ChatPromptTemplate
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(" ChatPromptTemplate RAG ready!")

 ChatPromptTemplate RAG ready!


In [10]:
# basic question
# my order has not arrived yet
# i went to request a refund
# thank you, issue resolved


In [27]:
# Chat Interface
def run_chat():
    print("\n" + "="*60)
    print("   TechCorp Customer Support 🤖")
    print("   Powered by LangChain + Ollama + ChromaDB (RAG)")
    print("="*60)
    print("  Commands:")
    print("  'tickets'        → see all tickets")
    print("  'close <TKT-ID>' → close a ticket")
    print("  'quit'           → exit")
    print("="*60)

    current_ticket = None

    while True:
        try:
            user_input = input("\nYou 😎: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nAgent 🤖 : Goodbye! I’ll just sit here... waiting 🥲")
            break

        if not user_input:
            continue

        # Commands
        if user_input.lower() == 'quit':
            print("\nAgent 🤖 : Logging off... but I’ll miss you 🥲 Goodbye!")
            break

        if user_input.lower() == 'tickets':
            show_tickets()
            continue

        if user_input.lower().startswith('close '):
            tid = user_input.split(' ', 1)[1].strip().upper()
            if tid in ticket_db:
                ticket_db[tid]['status'] = 'Resolved'
                print(f"\n🎉 Ticket {tid} marked as Resolved! Case closed like a pro 🕶️")
            else:
                print(f"\n😅 Ticket {tid} not found. Even I got confused!")
            continue

        # Thank you / Resolved
        if any(w in user_input.lower() for w in ['thanks','thank you','resolved','bye','done']):
            if current_ticket:
                ticket_db[current_ticket]['status'] = 'Resolved'
                print(f"\nAgent 🤖 : Mission accomplished 🎯")
                print(f"           Ticket {current_ticket} is now resolved!")
                print(f"           I deserve a raise for this 😎")
                current_ticket = None
            else:
                print("\nAgent 🤖 : You're welcome! I live to serve 😄")
            continue

        #  RAG Answer
        print("\nAgent 🤖 : Hmm... thinking harder than my salary allows 🧠💭...", end="\r")

        try:
            answer   = rag_chain.invoke(user_input)
            category = detect_category(user_input)

            # Create ticket on first message
            if not current_ticket:
                current_ticket = create_ticket(user_input, category)
                print(f"\n🎫 Boom! Ticket {current_ticket} created under '{category}'")
                print("   Relax, your problem is now officially my problem 😎")

            # Funny styles
            funny_prefix = random.choice([
                "Alright, here’s the scoop 🕵️:",
                "After intense thinking (3 seconds)... 🤔",
                "Good news! I found something 🥳:",
                "Breaking news from my brain 🧠:",
                "I did some AI magic ✨:"
            ])

            funny_suffix = random.choice([
                "Need anything else? I’m still here 😄",
                "Hope that helps! High-five? ✋",
                "Want more help? I don’t sleep 🤖",
                "Ask more, I’m bored anyway 😆",
                "Your wish = my command 😎"
            ])

            print(f"\nAgent 🤖 : {funny_prefix}")
            print(f"           {answer}")
            print(f"           {funny_suffix}")

            print(f"\n  [Ticket: {current_ticket} | Status: Open 🟢]")

        except Exception as e:
            print(f"\nAgent 🤖 : Uh-oh 😬 something broke inside me.")
            print("           Let’s pretend that didn’t happen and try again 😅")
            print(f"  [Error: {e}]")

# ── Run ────────────────────────────────────────────────────────
run_chat()


   TechCorp Customer Support 🤖
   Powered by LangChain + Ollama + ChromaDB (RAG)
  Commands:
  'tickets'        → see all tickets
  'close <TKT-ID>' → close a ticket
  'quit'           → exit



You 😎:  quit



Agent 🤖 : Logging off... but I’ll miss you 🥲 Goodbye!


# Whatsapp bat

In [13]:
pip install flask twilio

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 4.5 MB/s  0:00:0010.0 MB/s eta 0:00:0101
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [twilio]━━━━ 6/7 [twilio]
Note: you may need to restart the kernel to use updated packages.


I built SmartDesk — an AI-powered customer support system that uses Retrieval Augmented Generation to answer customer queries on WhatsApp in real time. The system loads a knowledge base from PDF documents, creates semantic embeddings using Ollama, stores them in ChromaDB vector database, and retrieves relevant context to generate accurate responses using LLaMA 3. It also has an automatic ticket creation system to track every customer issue."